# 🚀 BuildFail-SLM: 05 - Model Evaluation & Benchmark (Qwen2.5-Coder-7B-Instruct)

Evaluates our fine-tuned 7B model (**`krishna1707/buildfail-slm-qwen7b-adapter`**) directly from Hugging Face Hub (or local `./checkpoints_qwen7b`).

### Evaluation Protocol:
1. **In-Distribution Test Benchmark (`data/04_finetuning/test.jsonl`)**: Measures diagnosis accuracy & unified diff validity.
2. **Out-of-Distribution Benchmark (`data/04_finetuning/ood_test.jsonl`)**: Measures generalization on strictly unseen repositories across all 14 failure categories.

## 1. Inspect Benchmark Test Sets

In [1]:
import json
import re
from pathlib import Path
import pandas as pd

# Robust path resolver (works whether notebook runs from notebooks/ or project root)
FINETUNE_DIR = Path("../data/04_finetuning") if Path("../data/04_finetuning").exists() else Path("data/04_finetuning")
test_path = FINETUNE_DIR / "test.jsonl"
ood_path = FINETUNE_DIR / "ood_test.jsonl"

def validate_diff(text: str) -> bool:
    if not text or not isinstance(text, str):
        return False
    has_diff_block = "```diff" in text or "diff --git" in text
    has_minus = bool(re.search(r"^\s*-\s+", text, re.MULTILINE))
    has_plus = bool(re.search(r"^\s*\+\s+", text, re.MULTILINE))
    has_hunk = bool(re.search(r"@@ -\d+,\d+ \+\d+,\d+ @@", text))
    return (has_minus and has_plus) or has_diff_block or has_hunk

def extract_category(text: str) -> str:
    if not isinstance(text, str):
        return text.get("failure_type", text.get("failure_category", "Unknown")) if isinstance(text, dict) else "Unknown"
    m = re.search(r"###?\s*(?:Failure\s*Category|Failure\s*Type):\s*\n*(.+)", text, re.IGNORECASE)
    if m:
        return m.group(1).split("\n")[0].strip().strip("*` ")
    TAXONOMY = [
        "Docker / Container Error", "Authentication / Permission Error", "Dependency Error",
        "Test Failure", "Configuration Error", "Environment Error", "Compilation Error",
        "Runtime Error", "Infrastructure Error", "Network Error", "Deployment Error",
        "Build Error", "Code Error", "Unknown"
    ]
    for cat in sorted(TAXONOMY, key=len, reverse=True):
        if cat.lower() in text.lower():
            return cat
    return "Unknown"

def evaluate_split(file_path: Path):
    if not file_path.exists():
        return None
    total = 0
    diff_count = 0
    categories = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            total += 1
            d = json.loads(line)
            out = d.get("output", "")
            out_str = out if isinstance(out, str) else json.dumps(out)
            if validate_diff(out_str) or "diff" in out_str:
                diff_count += 1
            categories.append(extract_category(out_str))
            
    return {
        "Split": file_path.name,
        "Total Evaluated": total,
        "Samples with Proposed Diffs": diff_count,
        "Unique Categories Covered": len(set(categories))
    }

res_test = evaluate_split(test_path)
res_ood = evaluate_split(ood_path)

df_results = pd.DataFrame([res_test, res_ood])
display(df_results)


,Split,Total Evaluated,Samples with Proposed Diffs,Unique Categories Covered
0,test.jsonl,3302,20,12
1,ood_test.jsonl,489,4,12


## 2. Load Fine-Tuned Model Directly from Hugging Face Hub

In [2]:
import os
import torch
from pathlib import Path
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Native .env loader (zero external dependencies, avoiding missing dotenv package errors)
env_file = Path("../.env") if Path("../.env").exists() else Path(".env")
if env_file.exists():
    for line in env_file.read_text(encoding="utf-8").splitlines():
        if line.strip().startswith("HUGGINGFACE_TOKEN="):
            os.environ["HUGGINGFACE_TOKEN"] = line.split("=", 1)[1].strip("\"' ")

HF_TOKEN = os.getenv("HUGGINGFACE_TOKEN", "")

# HUGGING FACE URL OF OUR FINE-TUNED MODEL:
ADAPTER_REPO = "krishna1707/buildfail-slm-qwen7b-adapter"
BASE_MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"

print(f"Pulling fine-tuned adapter directly from: https://huggingface.co/{ADAPTER_REPO}...")

# 1. Load tokenizer directly from Hugging Face repository
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# 2. Load base model with 4-bit NF4 quantization
if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        token=HF_TOKEN,
        quantization_config=bnb_config,
        device_map={"": 0},
        trust_remote_code=True
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=torch.float32,
        device_map="cpu",
        trust_remote_code=True
    )

# 3. Attach our fine-tuned adapter weights from Hugging Face Hub
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO, token=HF_TOKEN)
model.eval()
print("🎉 Successfully loaded BuildFail-SLM Adapter from Hugging Face Model Hub!")


c:\Users\reddy\OneDrive\document\SLM fine tuning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pulling fine-tuned adapter directly from: https://huggingface.co/krishna1707/buildfail-slm-qwen7b-adapter...
Device: cuda:0


c:\Users\reddy\OneDrive\document\SLM fine tuning\.venv\Lib\site-packages\transformers\modeling_utils.py:5107: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  _ = torch.empty(int(byte_count // 2), dtype=torch.float16, device=device, requires_grad=False)
Loading weights: 100%|██████████| 339/339 [00:36<00:00,  9.37it/s]


🎉 Successfully loaded BuildFail-SLM Adapter from Hugging Face Model Hub!


## 3. Run Inference on Real Out-of-Distribution (OOD) CI/CD Failure Cases (All 14 Categories)

In [3]:
PROMPT_TEMPLATE = """<|im_start|>system
You are BuildFail-SLM, an expert AI DevOps and software repair model.
Analyze CI/CD failures accurately, identify root causes, and propose non-destructive unified diffs.
<|im_end|>
<|im_start|>user
Analyze the following CI/CD failure log, diagnose the root cause, classify the failure category according to the 14-class taxonomy, and propose a concrete code patch in unified diff format.

### Target Context:
- Target File: {source_file}
- Environment: {environment}

### Failure Log:
```
{failure_log}
```

Format your response exactly as:
### Failure Category:
<One of: Code Error, Dependency Error, Build Error, Test Failure, Configuration Error, Environment Error, Compilation Error, Runtime Error, Infrastructure Error, Authentication / Permission Error, Network Error, Deployment Error, Docker / Container Error, Unknown>

### Root Cause Analysis:
<Precise, technical explanation of why the build or test failed>

### Recommended Remediation:
<Remediation steps or terminal command, if applicable>

### Proposed Code Fix (Unified Diff):
```diff
--- a/{source_file}
+++ b/{source_file}
@@ ... @@
- <old code>
+ <fixed code>
```
<|im_end|>
<|im_start|>assistant
"""

sample_cases = [
    {
        "category": "Docker / Container Error",
        "source_file": "Dockerfile",
        "environment": "Docker / Linux",
        "failure_log": "Step 4/10 : RUN pip install -r requirements.txt\nERROR: Could not find a version that satisfies the requirement torch==2.6.0+cu124\nERROR: No matching distribution found for torch==2.6.0+cu124\nThe command '/bin/sh -c pip install -r requirements.txt' returned a non-zero code: 1"
    },
    {
        "category": "Authentication / Permission Error",
        "source_file": ".github/workflows/deploy.yml",
        "environment": "GitHub Actions",
        "failure_log": "fatal: could not read Username for 'https://github.com': No such device or address\nremote: Invalid username or password.\nError: Process completed with exit code 128 (403 Forbidden: Resource not accessible by personal access token)."
    },
    {
        "category": "Dependency Error",
        "source_file": "package.json",
        "environment": "Node.js 18",
        "failure_log": "npm error code ERESOLVE\nnpm error ERESOLVE could not resolve\nnpm error Conflicting peer dependency: react@18.2.0\nnpm error Found: react@19.0.0\nnpm error Fix the upstream dependency conflict, or use --force or --legacy-peer-deps"
    },
    {
        "category": "Test Failure",
        "source_file": "tests/test_auth.py",
        "environment": "pytest / Python 3.11",
        "failure_log": "FAILED tests/test_auth.py::test_login_missing_password - TypeError: authenticate() missing 1 required positional argument: 'password'\nassert 400 == 200\n=== 1 failed, 24 passed in 0.45s ==="
    },
    {
        "category": "Configuration Error",
        "source_file": ".github/workflows/ci.yml",
        "environment": "GitHub Actions Runner",
        "failure_log": "yaml: line 14: mapping values are not allowed in this context\nError: .github/workflows/ci.yml (Line: 14, Col: 11): Unexpected value 'on:'"
    },
    {
        "category": "Environment Error",
        "source_file": "src/config.ts",
        "environment": "Vite / React",
        "failure_log": "npm error app@1.0.0 build: vite build\nUncaught Error: Missing required environment variable: VITE_API_BASE_URL\n    at Object.<anonymous> (src/config.ts:8:11)"
    },
    {
        "category": "Compilation Error",
        "source_file": "src/main.rs",
        "environment": "Rust cargo 1.78",
        "failure_log": "error[E0308]: mismatched types\n  --> src/main.rs:24:18\n   |\n24 |     let port: u16 = std::env::var(\"PORT\").unwrap();\n   |                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^ expected `u16`, found `String`"
    },
    {
        "category": "Runtime Error",
        "source_file": "app/services/payment.py",
        "environment": "FastAPI / Python",
        "failure_log": "AttributeError: 'NoneType' object has no attribute 'stripe_customer_id'\n  File 'app/services/payment.py', line 45, in charge_user\n    charge = stripe.Charge.create(customer=user.stripe_customer_id, amount=1000)"
    },
    {
        "category": "Infrastructure Error",
        "source_file": "kubernetes/deployment.yml",
        "environment": "Kubernetes / Linux Node",
        "failure_log": "Pod failed to schedule: 0/4 nodes are available: 4 Insufficient memory.\nStatus: OOMKilled (Exit Code 137). Process terminated by Linux cgroup OOM-killer."
    },
    {
        "category": "Network Error",
        "source_file": "scripts/download_assets.sh",
        "environment": "cURL / Bash",
        "failure_log": "curl: (28) Failed to connect to s3.us-east-1.amazonaws.com port 443 after 30002 ms: Connection timed out\nERROR: Asset download pipeline failed with exit code 28."
    },
    {
        "category": "Deployment Error",
        "source_file": "helm/Chart.yaml",
        "environment": "Helm 3",
        "failure_log": "Error: UPGRADE FAILED: pre-upgrade hooks failed: warning: Hook pre-upgrade db-migrate-job failed: Job failed to complete within backoffLimit 3"
    },
    {
        "category": "Build Error",
        "source_file": "webpack.config.js",
        "environment": "Webpack 5",
        "failure_log": "Module not found: Error: Can't resolve 'path' in '/app/src'\nBREAKING CHANGE: webpack < 5 used to include polyfills for node.js core modules by default."
    },
    {
        "category": "Code Error",
        "source_file": "utils/parser.py",
        "environment": "Python 3.12",
        "failure_log": "  File 'utils/parser.py', line 12\n    return {k: v for k, v in data.items()\n                                         ^\nSyntaxError: closing parenthesis '}' does not match opening parenthesis '{'"
    },
    {
        "category": "Unknown",
        "source_file": "scripts/ci_runner.sh",
        "environment": "Bash",
        "failure_log": "Command terminated with unknown non-zero status signal 11 (SIGSEGV: Segmentation fault, core dumped)."
    }
]

device = "cuda:0" if torch.cuda.is_available() else "cpu"

for idx, case in enumerate(sample_cases, 1):
    print(f"\n{'='*75}")
    print(f"CASE {idx:02d}/14 | EXPECTED CATEGORY: [{case['category']}] | FILE: {case['source_file']}")
    print(f"{'='*75}")
    
    prompt = PROMPT_TEMPLATE.format(
        source_file=case["source_file"],
        environment=case["environment"],
        failure_log=case["failure_log"]
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.2,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(response.strip())



CASE 01/14 | EXPECTED CATEGORY: [Docker / Container Error] | FILE: Dockerfile
### Failure Category:
Docker / Container Error

### Root Cause Analysis:
pip cannot find torch==2.6.0+cu124

### Recommended Repair Command:
`docker login $REGISTRY && docker push $IMAGE:$TAG`

CASE 02/14 | EXPECTED CATEGORY: [Authentication / Permission Error] | FILE: .github/workflows/deploy.yml
### Failure Category:
Authentication / Permission Error

### Root Cause Analysis:
GitHub Actions deploy workflow failing with "could not read Username" error

### Recommended Remediation:
Add `git config --global credential.helper store` before git push

### Fixed Code Sample:
```yaml
# Add credential helper configuration before git push
steps:
  - name: Deploy to production
    run: |
      git config --global credential.helper store
      echo "https://$GH_TOKEN@github.com" | tee ~/.netrc
      git remote set-url origin https://github.com/myorg/myrepo.git
      git push origin main
    env:
      GH_TOKEN: ${{ se

## 4. Quantitative Benchmark: In-Distribution vs. Out-of-Distribution

In [4]:
import time

# Canonical benchmark prompt matching the dataset's exact ChatML structure
BENCH_PROMPT_TEMPLATE = """<|im_start|>system
You are BuildFail-SLM, an expert AI DevOps and software repair model.
Analyze CI/CD failures accurately, identify root causes, and propose non-destructive unified diffs.
<|im_end|>
<|im_start|>user
{instruction}

{input_text}
<|im_end|>
<|im_start|>assistant
"""

def run_benchmark_eval(jsonl_file: Path, max_eval: int = 25):
    """Evaluates accuracy, unified diff proposals, and latency across test samples."""
    correct_cat = 0
    valid_diffs = 0
    total = 0
    latencies = []
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    
    with open(jsonl_file, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            if idx >= max_eval:
                break
            total += 1
            d = json.loads(line)
            inst = d.get("instruction", "Diagnose the CI/CD failure root cause and propose a code fix:")
            inp = d.get("input", "")
            ground_truth_out = d.get("output", "")
            inp_str = inp if isinstance(inp, str) else json.dumps(inp)
            gt_str = ground_truth_out if isinstance(ground_truth_out, str) else json.dumps(ground_truth_out)
            
            expected_cat = extract_category(gt_str)
            
            prompt = BENCH_PROMPT_TEMPLATE.format(instruction=inst, input_text=inp_str[:1500])
            inputs = tokenizer(prompt, return_tensors="pt").to(device)
            
            t0 = time.time()
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=150,
                    temperature=0.1,
                    pad_token_id=tokenizer.eos_token_id
                )
            latencies.append(time.time() - t0)
            
            pred_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            pred_cat = extract_category(pred_text)
            
            if pred_cat != "Unknown" and (pred_cat.lower() in expected_cat.lower() or expected_cat.lower() in pred_cat.lower()):
                correct_cat += 1
            
            if validate_diff(pred_text) or "diff" in pred_text:
                valid_diffs += 1
                
    return {
        "Benchmark Split": jsonl_file.name,
        "Samples Tested": total,
        "Diagnosis Accuracy": f"{(correct_cat / total) * 100:.1f}%" if total else "N/A",
        "Unified Diff Synthesis Rate": f"{(valid_diffs / total) * 100:.1f}%" if total else "N/A",
        "Avg Latency (sec)": f"{sum(latencies) / len(latencies):.2f}s" if latencies else "N/A"
    }

print("Running quantitative evaluation on Test and OOD benchmarks...")
bench_test = run_benchmark_eval(test_path, max_eval=25)
bench_ood = run_benchmark_eval(ood_path, max_eval=25)

df_bench = pd.DataFrame([bench_test, bench_ood])
display(df_bench)
print("\n✅ Full Evaluation Pipeline Complete! Model verified on Hugging Face: https://huggingface.co/krishna1707/buildfail-slm-qwen7b-adapter")


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (746 > 512). Running this sequence through the model will result in indexing errors


Running quantitative evaluation on Test and OOD benchmarks...


,Benchmark Split,Samples Tested,Diagnosis Accuracy,Unified Diff Synthesis Rate,Avg Latency (sec)
0,test.jsonl,25,20.0%,0.0%,9.66s
1,ood_test.jsonl,25,80.0%,0.0%,8.94s



✅ Full Evaluation Pipeline Complete! Model verified on Hugging Face: https://huggingface.co/krishna1707/buildfail-slm-qwen7b-adapter
